# Taller Evaluativo 02 — Funciones y POO (Calidad del Aire PM2.5)

**Estudiante:** Wolfang Felipe Pérez Roldán  
**Fecha:** 13 de octubre de 2026

## Ejercicio 1 — Calibrar una lectura

Se define `calibrar_lectura(valor, pendiente=1.0, intercepto=0.0)`, que aplica la
recta de calibración `valor * pendiente + intercepto`. Los parámetros de calibración
son opcionales y sus valores por defecto están en la firma, así que sin argumentos la
lectura no se altera.

La función entrega el resultado con `return` (no imprime). Al final se calculan los
tres casos: sin calibrar, con pendiente e intercepto, y pasando solo el intercepto
por nombre.

In [1]:
def calibrar_lectura(valor, pendiente=1.0, intercepto=0.0):
    """Devuelve la lectura corregida por la recta de calibración."""
    return valor * pendiente + intercepto


sin_calibrar = calibrar_lectura(10.0)
calibrada = calibrar_lectura(10.0, 1.2, -0.5)
con_intercepto = calibrar_lectura(10.0, intercepto=2.0)

print("Sin calibrar:", sin_calibrar)
print("Calibrada:", calibrada)
print("Con intercepto:", con_intercepto)

Sin calibrar: 10.0
Calibrada: 11.5
Con intercepto: 12.0


## Ejercicio 2 — ¿Es utilizable la lectura?

Se declara `lecturas_sbc_a`, la lista con las lecturas crudas del sensor SBC-A (en
ug/m3), que incluye el centinela -999.0 y un valor fuera de rango.

Una lectura sirve si no es el centinela -999.0 y si está entre 0 y 500.0 inclusive.
`es_lectura_valida(valor)` devuelve ese booleano con `return`, no imprime nada y no lee
variables globales: el valor llega siempre por parámetro.

In [2]:
lecturas_sbc_a = [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


def es_lectura_valida(valor):
    """Indica si la lectura no es el centinela y está entre 0 y 500 ug/m3."""
    return valor != -999.0 and 0.0 <= valor <= 500.0


print("25.0   ->", es_lectura_valida(25.0))
print("-999.0 ->", es_lectura_valida(-999.0))
print("600.0  ->", es_lectura_valida(600.0))
print("0.0    ->", es_lectura_valida(0.0))

25.0   -> True
-999.0 -> False
600.0  -> False
0.0    -> True


## Ejercicio 3 — Promedio de las lecturas útiles

`promedio_valido(lecturas)` recorre la lista recibida con un `for` y acumula solo las
lecturas para las que `es_lectura_valida` devuelve `True`: la regla no se reescribe a
mano.

El acumulador y el contador son locales a la función y, si ninguna lectura es válida,
devuelve `None` en lugar de dividir entre cero.

In [3]:
def promedio_valido(lecturas):
    """Promedia solo las lecturas válidas; devuelve None si no hay ninguna."""
    suma = 0.0
    cantidad = 0

    for valor in lecturas:
        if es_lectura_valida(valor):
            suma += valor
            cantidad += 1

    if cantidad == 0:
        return None

    return suma / cantidad


promedio_sbc_a = promedio_valido(lecturas_sbc_a)

print("Promedio valido SBC-A:", promedio_sbc_a)
print("Promedio de solo centinelas:", promedio_valido([-999.0, -999.0]))

Promedio valido SBC-A: 27.0
Promedio de solo centinelas: None


## Ejercicio 4 — Calibrar la serie con map, filter y lambda

Con `filter` y `es_lectura_valida` se dejan solo las lecturas útiles de SBC-A, y con
`map` y una `lambda` se les aplica la calibración del sensor (pendiente 1.2 e
intercepto -0.5). Como `filter` y `map` devuelven objetos perezosos, el resultado se
convierte con `list(...)`.

Después se cuentan con `len` las lecturas calibradas que superan 35.4, el límite de la
categoría Moderada. En este ejercicio no se usa `for` ni comprensión de listas.

In [4]:
pendiente_sbc_a = 1.2
intercepto_sbc_a = -0.5

lecturas_utiles = list(filter(es_lectura_valida, lecturas_sbc_a))

lecturas_calibradas = list(
    map(lambda lectura: calibrar_lectura(lectura, pendiente_sbc_a, intercepto_sbc_a), lecturas_utiles)
)

excedencias = len(list(filter(lambda concentracion: concentracion > 35.4, lecturas_calibradas)))

print("Lecturas calibradas:", lecturas_calibradas)
print("Excedencias sobre 35.4:", excedencias)

Lecturas calibradas: [11.5, 29.5, 47.5, 9.1, 61.9]
Excedencias sobre 35.4: 2


## Ejercicio 5 — La clase Lectura

`Lectura` guarda el valor, la unidad y el código del sensor. `a_microgramos()` pasa la
concentración a ug/m3 (multiplica por 1000 si la unidad es "mg/m3") y
`es_dato_faltante()` avisa si el valor es el centinela -999.0.

En este ejercicio la clase no valida: el centinela es un valor legítimo que debe poder
guardarse. Los métodos leen los datos con `self` y los valores por defecto van en
`__init__`.

In [5]:
class Lectura:
    """Lectura de concentración con su unidad y su sensor de origen."""

    def __init__(self, valor, unidad="ug/m3", codigo_sensor="sin asignar"):
        self.valor = valor
        self.unidad = unidad
        self.codigo_sensor = codigo_sensor

    def a_microgramos(self):
        """Devuelve la concentración expresada en ug/m3."""
        if self.unidad == "mg/m3":
            return self.valor * 1000
        return self.valor

    def es_dato_faltante(self):
        """Devuelve True si el valor es el centinela de dato faltante."""
        return self.valor == -999.0


lectura_campo = Lectura(0.0252, "mg/m3", "SBC-A")
lectura_faltante = Lectura(-999.0, codigo_sensor="SBC-A")
lectura_normal = Lectura(30.0)

print("a_microgramos de 0.0252 mg/m3:", lectura_campo.a_microgramos())
print("lectura_faltante es dato faltante:", lectura_faltante.es_dato_faltante())
print("lectura_campo es dato faltante:", lectura_campo.es_dato_faltante())
print("lectura_normal:", lectura_normal.unidad, "-", lectura_normal.codigo_sensor)

a_microgramos de 0.0252 mg/m3: 25.2
lectura_faltante es dato faltante: True
lectura_campo es dato faltante: False
lectura_normal: ug/m3 - sin asignar


## Ejercicio 6 — La clase Sensor: validación y encapsulamiento

Un sensor con una pendiente de calibración inválida produciría resultados absurdos en
silencio, así que `__init__` lanza `ValueError` antes de guardar nada si el código es
un texto vacío o si la pendiente es menor o igual a cero.

El código queda como atributo público, pero la pendiente y el intercepto son internos
(`_pendiente` e `_intercepto`). `calibrar()` no repite la fórmula: reutiliza
`calibrar_lectura`. `__repr__` muestra el objeto como
`Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)`.

Las dos comprobaciones manuales de la validación quedan comentadas al final de la
celda, porque ambas deben fallar con `ValueError`.

In [6]:
class Sensor:
    """Sensor con su recta de calibración; impide pendientes inválidas."""

    def __init__(self, codigo, pendiente=1.0, intercepto=0.0):
        if codigo == "":
            raise ValueError("El código del sensor no puede ser un texto vacío.")
        if pendiente <= 0:
            raise ValueError("La pendiente de calibración debe ser mayor que cero.")
        self.codigo = codigo
        self._pendiente = pendiente
        self._intercepto = intercepto

    def calibrar(self, valor):
        """Devuelve la lectura calibrada reutilizando calibrar_lectura."""
        return calibrar_lectura(valor, self._pendiente, self._intercepto)

    def __repr__(self):
        return (f"Sensor(codigo={self.codigo!r}, pendiente={self._pendiente!r}, "
                f"intercepto={self._intercepto!r})")

    def __str__(self):
        return (f"Sensor {self.codigo}: pendiente={self._pendiente}, "
                f"intercepto={self._intercepto}")


sensor_a = Sensor("SBC-A", 1.2, -0.5)
sensor_b = Sensor("SBC-B")

print(sensor_a)
print([sensor_a, sensor_b])
print("Calibración de 10.0 con sensor_a:", sensor_a.calibrar(10.0))
print("Calibración de 10.0 con sensor_b:", sensor_b.calibrar(10.0))

# Comprobación manual de la validación (queda comentada en la entrega):
# Sensor("")      -> ValueError: el código no puede estar vacío
# Sensor("X", 0)  -> ValueError: la pendiente debe ser mayor que cero

Sensor SBC-A: pendiente=1.2, intercepto=-0.5
[Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5), Sensor(codigo='SBC-B', pendiente=1.0, intercepto=0.0)]
Calibración de 10.0 con sensor_a: 11.5
Calibración de 10.0 con sensor_b: 10.0


## Ejercicio 7 — La clase Estacion: objetos que contienen objetos

La estación EST-04 guarda sus lecturas como objetos `Lectura`. `agregar()` las va
metiendo en la lista interna y `valores()` devuelve las concentraciones en ug/m3, en
el mismo orden, llamando a `a_microgramos()` de cada lectura.

Con `datos_campana` (pares de valor y unidad, uno de ellos en mg/m3) se llena la
estación con un `for` y se comprueba que los valores coinciden con `lecturas_sbc_a`
del Ejercicio 2.

In [7]:
class Estacion:
    """Estación de monitoreo que guarda objetos Lectura."""

    def __init__(self, codigo, sector):
        self.codigo = codigo
        self.sector = sector
        self._lecturas = []

    def agregar(self, lectura):
        """Agrega una Lectura a la estación."""
        self._lecturas.append(lectura)

    def valores(self):
        """Devuelve las concentraciones en ug/m3, en el orden de ingreso."""
        return [lectura.a_microgramos() for lectura in self._lecturas]

    def __repr__(self):
        return f"Estacion({self.codigo!r}, {self.sector!r}, lecturas={len(self._lecturas)})"


datos_campana = [
    (10.0, "ug/m3"), (-999.0, "ug/m3"), (0.025, "mg/m3"), (40.0, "ug/m3"),
    (-999.0, "ug/m3"), (8.0, "ug/m3"), (52.0, "ug/m3"), (600.0, "ug/m3"),
]

estacion_oriente = Estacion("EST-04", "Oriente")

for valor, unidad in datos_campana:
    estacion_oriente.agregar(Lectura(valor, unidad, "SBC-A"))

print(estacion_oriente)
print(estacion_oriente.valores())

Estacion('EST-04', 'Oriente', lecturas=8)
[10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


## Ejercicio 8 — Integración: el informe de la estación

`categoria_pm25` clasifica la concentración con una sola cadena `if / elif / else`:
Buena hasta 9.0, Moderada hasta 35.4, Dañina para grupos sensibles hasta 55.4 y Dañina
para la salud por encima de 55.4.

`InformeEstacion` junta la estación con su sensor. `promedio_calibrado()` toma los
valores, descarta los que no sirven con `filter`, los calibra con `map` usando el
método `calibrar` del sensor y saca el promedio con `promedio_valido`. `__str__`
imprime el código de la estación, el promedio redondeado a un decimal y la categoría.

El reto de extensión va incluido en la misma clase: `excedencias()` cuenta con `filter`
y una `lambda` las lecturas calibradas que superan el umbral.

In [8]:
def categoria_pm25(concentracion):
    """Devuelve la categoría oficial de calidad del aire para PM2.5."""
    if concentracion <= 9.0:
        return "Buena"
    elif concentracion <= 35.4:
        return "Moderada"
    elif concentracion <= 55.4:
        return "Dañina para grupos sensibles"
    else:
        return "Dañina para la salud"


class InformeEstacion:
    """Informe que combina una Estación con el Sensor que la calibra."""

    def __init__(self, estacion, sensor):
        self._estacion = estacion
        self._sensor = sensor

    def promedio_calibrado(self):
        """Promedio de las lecturas útiles, ya calibradas por el sensor."""
        valores = self._estacion.valores()
        utiles = filter(es_lectura_valida, valores)
        calibradas = list(map(self._sensor.calibrar, utiles))
        return promedio_valido(calibradas)

    def categoria(self):
        """Categoría del promedio calibrado."""
        return categoria_pm25(self.promedio_calibrado())

    def excedencias(self, umbral=35.4):
        """Cuenta las lecturas calibradas que superan el umbral (reto de extensión)."""
        utiles = filter(es_lectura_valida, self._estacion.valores())
        calibradas = map(self._sensor.calibrar, utiles)
        return len(list(filter(lambda concentracion: concentracion > umbral, calibradas)))

    def __str__(self):
        return (f"{self._estacion.codigo}: {round(self.promedio_calibrado(), 1)} ug/m3"
                f" - {self.categoria()}")


informe = InformeEstacion(estacion_oriente, sensor_a)

print(informe)
print("Promedio calibrado:", informe.promedio_calibrado())
print("Excedencias sobre 35.4:", informe.excedencias())

EST-04: 31.9 ug/m3 - Moderada
Promedio calibrado: 31.9
Excedencias sobre 35.4: 2


In [9]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert abs(calibrar_lectura(10.0, 1.2, -0.5) - 11.5) < 1e-9, "revise el Ejercicio 1"
assert calibrar_lectura(10.0) == 10.0, "sin parámetros no debe alterar la lectura"
assert abs(con_intercepto - 12.0) < 1e-9, "revise con_intercepto del Ejercicio 1"
assert es_lectura_valida(25.0) and not es_lectura_valida(-999.0), "revise el Ejercicio 2"
assert not es_lectura_valida(600.0), "600.0 excede el rango del sensor"
assert abs(promedio_sbc_a - 27.0) < 1e-9, "revise el promedio del Ejercicio 3"
assert promedio_valido([-999.0, -999.0]) is None, "sin lecturas válidas debe devolver None"
assert len(lecturas_calibradas) == 5, "deberían quedar 5 lecturas calibradas"
assert abs(lecturas_calibradas[0] - 11.5) < 1e-9, "revise la calibración del Ejercicio 4"
assert excedencias == 2, "deberían ser 2 lecturas sobre 35.4"
assert abs(lectura_campo.a_microgramos() - 25.2) < 1e-9, "revise a_microgramos"
assert lectura_faltante.es_dato_faltante() and not lectura_campo.es_dato_faltante()
assert repr(sensor_a) == "Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)", "revise __repr__ de Sensor"
assert hasattr(sensor_a, "_pendiente") and not hasattr(sensor_a, "pendiente"), "la pendiente debe ser interna"
assert abs(sensor_b.calibrar(10.0) - 10.0) < 1e-9, "revise los valores por defecto de Sensor"
assert repr(estacion_oriente) == "Estacion('EST-04', 'Oriente', lecturas=8)", "revise __repr__ de Estacion"
assert estacion_oriente.valores() == lecturas_sbc_a, "revise valores() del Ejercicio 7"
assert abs(informe.promedio_calibrado() - 31.9) < 1e-9, "revise el Ejercicio 8"
assert informe.categoria() == "Moderada", "revise categoria()"
assert str(informe) == "EST-04: 31.9 ug/m3 - Moderada", "revise __str__ de InformeEstacion"
print("Verificación completada sin errores.")

Verificación completada sin errores.
